# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/HSB-25/flyrank-ml-internship-W1-Run-the-Starter-Notebooks/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

One row = one content item's Search Console performance on one day: the grain is
(client_hash_id, content_hash_id, report_date) in `fact_content_daily_performance`.
This carries forward my Week-2 lane (Refresh / Content Opportunity Scoring) from the
content-item level onto the full warehouse.

Table(s): fact_content_daily_performance (daily fact, partitioned by month) as the
primary table, joined to dim_content for static metadata, dim_clients for
access/availability flags, and fact_content_query_90d for query-mix signals.

Time window: month=2026-03 (mid-panel, dev month). The final month (June 2026, exposed
separately as fact_content_daily_performance_sample.parquet) is sealed as the test
month ,it's the natural outcome window for any past future label, so I never touch it
while building label logic.

Predict/rank: is_declining whether a content item's impressions in the second half
of the month drop more than 20% vs. the first half. A proxy for "should this page
enter the review queue," not a causal claim.

Deliberately excluded: any second-half aggregate used to derive the label ;that IS
the label in disguise, demonstrated and removed in Section 3.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one , typing sentences here breaks Run All.
import os, getpass, duckdb, numpy as np, pandas as pd

HF_TOKEN = os.environ.get('HF_TOKEN')
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get('HF_TOKEN')
    except Exception:
        pass
HF_TOKEN = HF_TOKEN or getpass.getpass('Paste your Hugging Face READ token (hf_...): ')

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
MONTH_PATH = f"{REL}/fact_content_daily_performance/month=2026-03/*.parquet"
TABLES = {
    'dim_clients':    f"read_parquet('{REL}/dim_clients.parquet')",
    'dim_content':    f"read_parquet('{REL}/dim_content.parquet')",
    'fact_query_90d': f"read_parquet('{REL}/fact_content_query_90d.parquet')",
}

# Query 1 — grain check: an empty result means no duplicate keys exist -> grain holds
grain_check = con.sql(f"""
    SELECT client_hash_id, content_hash_id, report_date, COUNT(*) AS n
    FROM read_parquet('{MONTH_PATH}')
    GROUP BY 1, 2, 3
    HAVING COUNT(*) > 1
""").df()
print('Rows violating stated grain (should be 0):', len(grain_check))
grain_check.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Rows violating stated grain (should be 0): 0


,client_hash_id,content_hash_id,report_date,n


## 2. Fields: feature / label / context / excluded

FEATURE fields (built from the first half of month=2026-03, before the decision cutoff):
- imp_first_half, clk_first_half, pos_first_half , from fact_content_daily_performance,
  days 1–15 only
- visible_queries, rare_share  from fact_content_query_90d, describes how the page
  earns traffic, not the outcome

LABEL field:
- is_declining , 1 if second-half impressions (days 16 end) are more than 20% below
  first-half impressions, else 0

CONTEXT fields (identifiers, not model inputs):
- client_hash_id, content_hash_id, report_date

EXCLUDED:
- Any second-half aggregate / percent-change column  it IS the label, not a feature
  (leakage lesson from notebook 02)
- Rows with first-half impressions < 100  CTR and position are noisy/undefined at
  near-zero volume
- Rows from clients that are inactive or lack confirmed GSC access , unreliable
  signal for this lane, filtered in Section 3

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

schema = con.sql(f"DESCRIBE SELECT * FROM read_parquet('{MONTH_PATH}')").df()
schema

,column_name,column_type,null,key,default,extra
0,report_date,DATE,YES,None,None,None
1,client_hash_id,VARCHAR,YES,None,None,None
2,content_hash_id,VARCHAR,YES,None,None,None
3,client_has_gsc,BOOLEAN,YES,None,None,None
4,client_has_ga4,BOOLEAN,YES,None,None,None
5,gsc_data_available,BOOLEAN,YES,None,None,None
6,ga4_data_available,BOOLEAN,YES,None,None,None
7,gsc_impressions,BIGINT,YES,None,None,None
8,gsc_clicks,BIGINT,YES,None,None,None
9,gsc_sum_position,BIGINT,YES,None,None,None


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

Grain was already verified in Section 1 (zero violating rows). Below: row count +
date span, availability filtered with IS TRUE, the five-feature frame, and the
deliberate-leakage trap from notebook 02, performed on real warehouse data.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import roc_auc_score

# Query 2 — row count + date span
span = con.sql(f"""
    SELECT COUNT(*) AS n_rows, COUNT(DISTINCT content_hash_id) AS n_content_items,
           COUNT(DISTINCT client_hash_id) AS n_clients,
           MIN(report_date) AS min_date, MAX(report_date) AS max_date
    FROM read_parquet('{MONTH_PATH}')
""").df()
print(span)

# Query 3 — availability, IS TRUE (excludes both false and null)
before = con.sql(f"SELECT COUNT(*) FROM read_parquet('{MONTH_PATH}')").fetchone()[0]
after = con.sql(f"""
    SELECT COUNT(*) FROM read_parquet('{MONTH_PATH}') f
    JOIN {TABLES['dim_clients']} c ON f.client_hash_id = c.client_hash_id
    WHERE c.is_active IS TRUE AND c.has_gsc_access IS TRUE
""").fetchone()[0]
print(f"before availability filter: {before:,} rows")
print(f"after  availability filter: {after:,} rows  ({after/before:.1%} survive)")

# Five features
features = con.sql(f"""
    WITH daily AS (
        SELECT client_hash_id, content_hash_id,
               SUM(CASE WHEN report_date <= DATE '2026-03-15' THEN gsc_impressions ELSE 0 END) AS imp_first_half,
               SUM(CASE WHEN report_date <= DATE '2026-03-15' THEN gsc_clicks ELSE 0 END) AS clk_first_half,
               AVG(CASE WHEN report_date <= DATE '2026-03-15' THEN gsc_avg_position END) AS pos_first_half,
               SUM(CASE WHEN report_date > DATE '2026-03-15' THEN gsc_impressions ELSE 0 END) AS imp_second_half
        FROM read_parquet('{MONTH_PATH}')
        GROUP BY 1, 2

        HAVING imp_first_half >= 100
    )
    SELECT * FROM daily
""").df()

qsignals = con.sql(f"""
    SELECT content_hash_id,
           ANY_VALUE(content_visible_query_count) AS visible_queries,
           ANY_VALUE(rare_impressions_share) AS rare_share
    FROM {TABLES['fact_query_90d']} GROUP BY content_hash_id
""").df()

frame = features.merge(qsignals, on='content_hash_id', how='left')
print(len(frame), 'content items in feature frame')

# Feature justifications ("knowable at the decision moment because..."):
# imp_first_half   -> only sums days 1-15, strictly before the decision cutoff
# clk_first_half   -> bounded to the same pre-decision window
# pos_first_half   -> averaged only over days 1-15
# visible_queries  -> trailing-90d query-mix snapshot, not derived from the label window
# rare_share       -> same reasoning, describes query composition not the outcome

# The trap
frame['is_declining'] = (frame['imp_second_half'] < 0.8 * frame['imp_first_half']).astype(int)
feat_cols = ['imp_first_half', 'clk_first_half', 'pos_first_half', 'visible_queries', 'rare_share']
X, y = frame[feat_cols].fillna(0), frame['is_declining']

honest = DecisionTreeClassifier(max_depth=3, random_state=42).fit(X, y)
honest_score = roc_auc_score(y, honest.predict_proba(X)[:, 1])
print(f"Honest AUC (five features only): {honest_score:.3f}")

frame['leak_pct_change'] = (frame['imp_second_half'] - frame['imp_first_half']) / frame['imp_first_half']

X_leaky = frame[feat_cols + ['leak_pct_change']].fillna(0)
leaky = DecisionTreeClassifier(max_depth=3, random_state=42).fit(X_leaky, y)
leaky_score = roc_auc_score(y, leaky.predict_proba(X_leaky)[:, 1])
print(f"Leaky AUC (+ leak_pct_change): {leaky_score:.3f}  <- jumps toward 1.0")
print("Why: leak_pct_change IS how is_declining was computed, so the tree just re-derives the label.")

frame = frame.drop(columns=['leak_pct_change'])
print(f"\nKept honest AUC: {honest_score:.3f}")

    n_rows  n_content_items  n_clients   min_date   max_date
0  9841378           331437         55 2026-03-01 2026-03-31
before availability filter: 9,841,378 rows
after  availability filter: 7,856,284 rows  (79.8% survive)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

77540 content items in feature frame
Honest AUC (five features only): 0.642
Leaky AUC (+ leak_pct_change): 1.000  <- jumps toward 1.0
Why: leak_pct_change IS how is_declining was computed, so the tree just re-derives the label.

Kept honest AUC: 0.642


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

Named limitation: unbalanced panel history + incomplete client dimension.
dim_clients.gsc_data_start varies widely per client (some from early 2025, some
mid-2026), so month=2026-03 reflects very different amounts of prior history per
client — thinner history means noisier first-half/second-half comparisons.
Separately, some client_hash_ids carry access_profile =
"source_only_missing_client_dimension" with null is_active/has_gsc_access, which is
exactly why the availability filter in Section 3 matters. This data also can't say
why impressions moved — only that they did — so every claim here stays observational
and decision-support, never causal.

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
history_spread = con.sql(f"""
    SELECT MIN(gsc_data_start) AS earliest_start, MAX(gsc_data_start) AS latest_start,
           COUNT(*) FILTER (WHERE gsc_data_start IS NULL) AS null_start_count,
           COUNT(*) AS n_clients_with_gsc
    FROM {TABLES['dim_clients']}
    WHERE has_gsc_access IS TRUE
""").df()
history_spread

,earliest_start,latest_start,null_start_count,n_clients_with_gsc
0,2025-01-27,2026-06-02,7,67


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.